<a href="https://colab.research.google.com/github/adamgilbert516/higher-ed-clustering/blob/main/Higher_Ed_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Clustering U.S. Colleges by Cost, Aid & Graduation Outcomes

**Author:** Adam Gilbert  
**Tools Used:** Python, Pandas, SQLite, Plotly, Scikit-Learn, Tableau


## Project Summary
This notebook explores financial aid patterns, tuition structures, and student outcomes in U.S. higher education. The goal is to understand how institutional characteristics — such as control type and cost — relate to graduation rates, retention, and loan default outcomes.

We aim to answer:
- How do tuition and net price vary across institution types?
- Who receives Pell Grants and federal loans?
- How does financial burden relate to student success?
- Can we model graduation rates based on financial indicators?


---

## Data Loading & Cleaning

We begin by selecting relevant variables from the dataset, cleaning column names, and ensuring consistent formatting. Missing values are handled appropriately to avoid bias in our analysis.

In [103]:
from google.colab import drive
drive.mount('/content/drive')

data_path = '/content/drive/My Drive/Colab Notebooks/data/'
import os

# List files in your data directory
print(os.listdir(data_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['MERGED1996_97_PP.csv', 'MERGED1997_98_PP.csv', 'MERGED1998_99_PP.csv', 'MERGED1999_00_PP.csv', 'MERGED2000_01_PP.csv', 'MERGED2001_02_PP.csv', 'MERGED2002_03_PP.csv', 'MERGED2003_04_PP.csv', 'MERGED2004_05_PP.csv', 'MERGED2005_06_PP.csv', 'MERGED2006_07_PP.csv', 'MERGED2007_08_PP.csv', 'MERGED2008_09_PP.csv', 'MERGED2009_10_PP.csv', 'MERGED2010_11_PP.csv', 'MERGED2011_12_PP.csv', 'MERGED2012_13_PP.csv', 'MERGED2013_14_PP.csv', 'MERGED2014_15_PP.csv', 'MERGED2015_16_PP.csv', 'MERGED2016_17_PP.csv', 'MERGED2017_18_PP.csv', 'MERGED2018_19_PP.csv', 'MERGED2019_20_PP.csv', 'MERGED2020_21_PP.csv', 'MERGED2021_22_PP.csv', 'MERGED2022_23_PP.csv', 'FieldOfStudyData1415_1516_PP.csv', 'FieldOfStudyData1516_1617_PP.csv', 'FieldOfStudyData1617_1718_PP.csv', 'FieldOfStudyData1718_1819_PP.csv', 'FieldOfStudyData1819_1920_PP.csv', 'FieldOfStudyData1920_2021_PP.csv', 'MERGE

In [104]:
# Load the most recent institution-level data
import pandas as pd

institution_df = pd.read_csv(data_path + 'Most-Recent-Cohorts-Institution.csv')

<ipython-input-104-1495f382b06e>:4: DtypeWarning:

Columns (9,1407,1408,1431,1432,1532,1537,1538,1539,1540,1542,1546,1589,1601,1602,1606,1608,1611,1614,1615,1616,1619,1620,1621,1622,1623,1624,1625,1626,1627,1628,1629,1653,1679,1690,1692,1697,1700,1702,1725,1726,1727,1728,1729,1743,1815,1816,1817,1818,1823,1824,1830,1831,1879,1880,1881,1882,1883,1884,1885,1886,1887,1888,1889,1890,1891,1892,1893,1894,1895,1896,1897,1898,1909,1910,1911,1912,1913,1957,1958,1959,1960,1961,1962,1963,1964,1965,1966,1967,1968,1969,1970,1971,1972,1973,1974,1975,1976,1983,1984,2376,2377,2403,2404,2495,2496,2497,2498,2499,2500,2501,2502,2503,2504,2505,2506,2507,2508,2509,2510,2511,2512,2513,2514,2515,2516,2517,2518,2519,2520,2521,2522,2523,2524,2525,2526,2527,2528,2529,2530,2958,3215,3231,3235,3236) have mixed types. Specify dtype option on import or set low_memory=False.



In [105]:
# Check shape and first few rows
print(f"Dataset Shape: {institution_df.shape}")
institution_df.head()


Dataset Shape: (6429, 3306)


,UNITID,OPEID,OPEID6,INSTNM,CITY,STABBR,ZIP,ACCREDAGENCY,INSTURL,NPCURL,...,COUNT_WNE_MALE1_P11,GT_THRESHOLD_P11,MD_EARN_WNE_INC1_P11,MD_EARN_WNE_INC2_P11,MD_EARN_WNE_INC3_P11,MD_EARN_WNE_INDEP0_P11,MD_EARN_WNE_INDEP1_P11,MD_EARN_WNE_MALE0_P11,MD_EARN_WNE_MALE1_P11,SCORECARD_SECTOR
0,100654,100200.0,1002.0,Alabama A & M University,Normal,AL,35762,Southern Association of Colleges and Schools C...,www.aamu.edu/,www.aamu.edu/admissions-aid/tuition-fees/net-p...,...,777.0,0.6250,36650.0,41070.0,47016.0,38892.0,41738.0,38167.0,40250.0,4
1,100663,105200.0,1052.0,University of Alabama at Birmingham,Birmingham,AL,35294-0110,Southern Association of Colleges and Schools C...,https://www.uab.edu/,https://tcc.ruffalonl.com/University of Alabam...,...,1157.0,0.7588,47182.0,51896.0,54368.0,50488.0,51505.0,46559.0,59181.0,4
2,100690,2503400.0,25034.0,Amridge University,Montgomery,AL,36117-3553,Southern Association of Colleges and Schools C...,https://www.amridgeuniversity.edu/,https://www2.amridgeuniversity.edu:9091/,...,67.0,0.5986,35752.0,41007.0,NaN,NaN,38467.0,32654.0,49435.0,5
3,100706,105500.0,1055.0,University of Alabama in Huntsville,Huntsville,AL,35899,Southern Association of Colleges and Schools C...,www.uah.edu/,finaid.uah.edu/,...,802.0,0.7810,51208.0,62219.0,62577.0,55920.0,60221.0,47787.0,67454.0,4
4,100724,100500.0,1005.0,Alabama State University,Montgomery,AL,36104-0271,Southern Association of Colleges and Schools C...,www.alasu.edu/,www.alasu.edu/cost-aid/tuition-costs/net-price...,...,1049.0,0.5378,32844.0,36932.0,37966.0,34294.0,31797.0,32303.0,36964.0,4


In [106]:
# List all columns to identify useful ones
institution_df.columns.tolist()

['UNITID',
 'OPEID',
 'OPEID6',
 'INSTNM',
 'CITY',
 'STABBR',
 'ZIP',
 'ACCREDAGENCY',
 'INSTURL',
 'NPCURL',
 'SCH_DEG',
 'HCM2',
 'MAIN',
 'NUMBRANCH',
 'PREDDEG',
 'HIGHDEG',
 'CONTROL',
 'ST_FIPS',
 'REGION',
 'LOCALE',
 'LOCALE2',
 'LATITUDE',
 'LONGITUDE',
 'CCBASIC',
 'CCUGPROF',
 'CCSIZSET',
 'HBCU',
 'PBI',
 'ANNHI',
 'TRIBAL',
 'AANAPII',
 'HSI',
 'NANTI',
 'MENONLY',
 'WOMENONLY',
 'RELAFFIL',
 'ADM_RATE',
 'ADM_RATE_ALL',
 'SATVR25',
 'SATVR75',
 'SATMT25',
 'SATMT75',
 'SATWR25',
 'SATWR75',
 'SATVRMID',
 'SATMTMID',
 'SATWRMID',
 'ACTCM25',
 'ACTCM75',
 'ACTEN25',
 'ACTEN75',
 'ACTMT25',
 'ACTMT75',
 'ACTWR25',
 'ACTWR75',
 'ACTCMMID',
 'ACTENMID',
 'ACTMTMID',
 'ACTWRMID',
 'SAT_AVG',
 'SAT_AVG_ALL',
 'PCIP01',
 'PCIP03',
 'PCIP04',
 'PCIP05',
 'PCIP09',
 'PCIP10',
 'PCIP11',
 'PCIP12',
 'PCIP13',
 'PCIP14',
 'PCIP15',
 'PCIP16',
 'PCIP19',
 'PCIP22',
 'PCIP23',
 'PCIP24',
 'PCIP25',
 'PCIP26',
 'PCIP27',
 'PCIP29',
 'PCIP30',
 'PCIP31',
 'PCIP38',
 'PCIP39',
 'PCIP40',

In [107]:
selected_columns = [
    # Institution Info
    'INSTNM', 'CITY', 'STABBR', 'ZIP', 'CONTROL',

    # Financial Cost
    'TUITIONFEE_IN', 'TUITIONFEE_OUT', 'COSTT4_A',
    'NPT4_PUB', 'NPT4_PRIV', 'TUITIONFEE_PROG',

    # Financial Aid
    'PCTPELL', 'PCTFLOAN', 'DEBT_MDN', 'GRAD_DEBT_MDN_SUPP',

    # Academic & Admissions Profile
    'ADM_RATE', 'SAT_AVG', 'AVGFACSAL',

    # Enrollment & Outcomes
    'UGDS', 'C150_4', 'RET_FT4', 'CDR3',
    'MD_EARN_WNE_P10'
]

aid_df = institution_df[selected_columns]

# Preview the cleaned dataframe
aid_df.head()


,INSTNM,CITY,STABBR,ZIP,CONTROL,TUITIONFEE_IN,TUITIONFEE_OUT,COSTT4_A,NPT4_PUB,NPT4_PRIV,...,DEBT_MDN,GRAD_DEBT_MDN_SUPP,ADM_RATE,SAT_AVG,AVGFACSAL,UGDS,C150_4,RET_FT4,CDR3,MD_EARN_WNE_P10
0,Alabama A & M University,Normal,AL,35762,1,10024.0,18634.0,23751.0,14559.0,NaN,...,16600,31000,0.6622,947.0,8610.0,5726.0,0.2874,0.6387,0.0,40628.0
1,University of Alabama at Birmingham,Birmingham,AL,35294-0110,1,8832.0,21864.0,27826.0,17727.0,NaN,...,15832,22300,0.8842,1251.0,12211.0,12118.0,0.6260,0.8195,0.0,54501.0
2,Amridge University,Montgomery,AL,36117-3553,2,NaN,NaN,NaN,NaN,NaN,...,13385,32189,NaN,NaN,5109.0,226.0,0.4000,NaN,0.0,37621.0
3,University of Alabama in Huntsville,Huntsville,AL,35899,1,11770.0,24662.0,27098.0,19880.0,NaN,...,13905,20705,0.7425,1321.0,10411.0,6650.0,0.6191,0.8050,0.0,61767.0
4,Alabama State University,Montgomery,AL,36104-0271,1,11248.0,19576.0,22028.0,13889.0,NaN,...,17500,31000,0.9564,977.0,8015.0,3322.0,0.3018,0.6045,0.0,34502.0


In [108]:
# Export the cleaned dataframe to CSV
aid_df.to_csv('/content/drive/My Drive/Colab Notebooks/data/financial_aid_analysis.csv', index=False)

# ✅ Ensure 'Institution_Type' is defined before grouping
if 'Institution_Type' not in aid_df.columns:
    aid_df = aid_df.copy()
    aid_df['Institution_Type'] = aid_df['CONTROL'].map({
        1: 'Public',
        2: 'Private Non-Profit',
        3: 'Private For-Profit'
    })



In [109]:
import sqlite3

# Create in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Load dataframe into SQL table
aid_df.to_sql('financial_aid', conn, index=False)

# Example Query 1: Average Tuition by Institution Type
query1 = """
SELECT Institution_Type,
       AVG(TUITIONFEE_IN) AS Avg_InState_Tuition,
       AVG(TUITIONFEE_OUT) AS Avg_OutState_Tuition
FROM financial_aid
GROUP BY Institution_Type
"""
pd.read_sql_query(query1, conn)

# Example Query 2: Identify high-risk schools
query2 = """
SELECT INSTNM, Institution_Type, PCTFLOAN, C150_4
FROM financial_aid
WHERE PCTFLOAN > 0.8 AND C150_4 < 0.4
ORDER BY PCTFLOAN DESC
LIMIT 10
"""
pd.read_sql_query(query2, conn)


,INSTNM,Institution_Type,PCTFLOAN,C150_4
0,Polytechnic University of Puerto Rico-Miami,Private Non-Profit,1.0000,0.0000
1,Pennsylvania Institute of Technology,Private Non-Profit,0.9186,0.3258
2,Cleveland University-Kansas City,Private Non-Profit,0.9111,0.3333
3,Remington College-Shreveport Campus,Private Non-Profit,0.9034,0.2593
4,Bloomfield College,Private Non-Profit,0.9002,0.3105
5,Remington College-Dallas Campus,Private Non-Profit,0.8917,0.3762
6,William Penn University,Private Non-Profit,0.8818,0.2924
7,Carolina Christian College,Private Non-Profit,0.8723,0.0000
8,Herzing University-Akron,Private Non-Profit,0.8645,0.2121
9,Bolivar Technical College,Private Non-Profit,0.8615,0.0000


## Tuition & Net Price by Institution Type

This section compares in-state vs. out-of-state tuition across institution types. We also examine net price — what students actually pay after financial aid — for both public and private institutions.

**Key Questions:**
- Are tuition and net price aligned across sectors?
- Do public institutions consistently offer lower net costs?


In [110]:
# Recreate Institution_Type column (if needed)
control_map = {1: 'Public', 2: 'Private Non-Profit', 3: 'Private For-Profit'}
aid_df['Institution_Type'] = aid_df['CONTROL'].map(control_map)

# Group by Institution_Type and calculate means
summary_stats = aid_df.groupby('Institution_Type')[[
    'TUITIONFEE_IN', 'TUITIONFEE_OUT', 'NPT4_PUB', 'NPT4_PRIV', 'PCTPELL', 'PCTFLOAN'
]].mean()


In [111]:
import plotly.express as px

# Melt and rename for clearer labels
tuition_df = summary_stats.reset_index()[['Institution_Type', 'TUITIONFEE_IN', 'TUITIONFEE_OUT']].melt(
    id_vars='Institution_Type',
    value_vars=['TUITIONFEE_IN', 'TUITIONFEE_OUT'],
    var_name='Tuition Type',
    value_name='Average Tuition ($)'
)

# Replace values for better display
tuition_df['Institution Type'] = tuition_df['Institution_Type']
tuition_df['Tuition Type'] = tuition_df['Tuition Type'].replace({
    'TUITIONFEE_IN': 'In-State Tuition',
    'TUITIONFEE_OUT': 'Out-of-State Tuition'
})

fig = px.bar(
    tuition_df,
    x='Institution Type',
    y='Average Tuition ($)',
    color='Tuition Type',
    barmode='group',
    title='Average In-State vs. Out-of-State Tuition by Institution Type'
)
fig.show()



In [112]:
# Step 1: Ensure Institution_Type exists
control_map = {1: 'Public', 2: 'Private Non-Profit', 3: 'Private For-Profit'}
aid_df['Institution_Type'] = aid_df['CONTROL'].map(control_map)

# Step 2: Create new column for Net Price depending on institution type
aid_df['Net_Price'] = aid_df.apply(
    lambda row: row['NPT4_PUB'] if row['Institution_Type'] == 'Public'
    else row['NPT4_PRIV'],
    axis=1
)

# Step 3: Group by Institution_Type and calculate average net price
net_price_summary = aid_df.groupby('Institution_Type')['Net_Price'].mean().reset_index()


In [113]:
# Prepare and rename
net_price_df = net_price_summary.reset_index()
net_price_df['Institution Type'] = net_price_df['Institution_Type']

fig = px.bar(
    net_price_df,
    x='Institution Type',
    y='Net_Price',
    title='Average Net Price by Institution Type',
    labels={'Net_Price': 'Average Net Price ($)'}
)
fig.show()



## Student Aid Patterns

Here, we explore the proportion of students receiving Pell Grants and federal loans at different institution types. These metrics give insight into the socioeconomic profiles of enrolled students.

**Key Questions:**
- Which institutions serve the most low-income students?
- Are there patterns of over-reliance on federal loans?


In [114]:
# Melt and rename for clarity
pell_loan_df = summary_stats.reset_index()[['Institution_Type', 'PCTPELL', 'PCTFLOAN']].melt(
    id_vars='Institution_Type',
    value_vars=['PCTPELL', 'PCTFLOAN'],
    var_name='Aid Type',
    value_name='Percentage of Students'
)

pell_loan_df['Institution Type'] = pell_loan_df['Institution_Type']
pell_loan_df['Aid Type'] = pell_loan_df['Aid Type'].replace({
    'PCTPELL': '% Receiving Pell Grants',
    'PCTFLOAN': '% Receiving Federal Loans'
})

fig = px.bar(
    pell_loan_df,
    x='Institution Type',
    y='Percentage of Students',
    color='Aid Type',
    barmode='group',
    title='% of Students Receiving Pell Grants and Federal Loans'
)
fig.show()


## Cost vs. Student Outcomes

We examine whether higher institutional costs correlate with better graduation and retention rates. Scatter plots and correlation metrics help us assess the strength and direction of these relationships.

**Key Questions:**
- Does spending more lead to better outcomes?
- Is there a clear return on educational investment?


In [115]:
# Drop missing values safely and avoid chained assignment warning
aid_df = aid_df.dropna(subset=['PCTFLOAN', 'CDR3']).copy()

# Compute correlation between loan rates and default rates by institution type
correlation_summary = (
    aid_df
    .groupby('Institution_Type')[['PCTFLOAN', 'CDR3']]
    .apply(lambda df: df['PCTFLOAN'].corr(df['CDR3']))
)

# Display result
print(correlation_summary)



Institution_Type
Private For-Profit   -0.057661
Private Non-Profit   -0.184673
Public               -0.054766
dtype: float64


In [116]:
# Prepare clean data
outcomes_df = aid_df[['Institution_Type', 'COSTT4_A', 'C150_4', 'RET_FT4']].copy().dropna()

# Correlation
grad_corr = outcomes_df['COSTT4_A'].corr(outcomes_df['C150_4'])
print(f"Correlation between Cost and Graduation Rate: {grad_corr:.2f}")

# Plotly scatter
fig = px.scatter(
    outcomes_df,
    x='COSTT4_A',
    y='C150_4',
    color='Institution_Type',
    title='Total Cost of Attendance vs. Graduation Rate',
    labels={
        'Institution_Type': 'Institution Type',
        'COSTT4_A': 'Total Cost of Attendance ($)',
        'C150_4': 'Graduation Rate'
    },
    opacity=0.6
)
fig.update_layout(yaxis_range=[0, 1])
fig.show()



Correlation between Cost and Graduation Rate: 0.58


In [117]:
# Correlation
ret_corr = outcomes_df['COSTT4_A'].corr(outcomes_df['RET_FT4'])
print(f"Correlation between Cost and Retention Rate: {ret_corr:.2f}")

# Plotly scatter
fig = px.scatter(
    outcomes_df,
    x='COSTT4_A',
    y='RET_FT4',
    color='Institution_Type',
    title='Total Cost of Attendance vs. Retention Rate',
    labels={
        'Institution_Type': 'Institution Type',
        'COSTT4_A': 'Total Cost of Attendance ($)',
        'RET_FT4': 'Retention Rate'
    },
    opacity=0.6
)
fig.update_layout(yaxis_range=[0, 1])
fig.show()


Correlation between Cost and Retention Rate: 0.37


## K-Means Clustering: Grouping Institutions by Financial & Outcome Indicators

We use K-Means clustering to segment institutions into distinct groups based on financial accessibility (% Pell Grants, % Federal Loans, Total Cost) and student success outcomes (Graduation Rate and Default Rate). Clusters are then labeled according to average graduation rate to reflect institutional profiles:
- **High-Risk, High-Aid Institutions**
- **Middle-Tier, Mixed-Aid Institutions**
- **Elite, Low-Aid Institutions**

The cluster assignments are added to the dataset for visualization and further analysis.


In [118]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from pandas.api.types import CategoricalDtype
import plotly.express as px

# Step 1: Select clustering features and keep institution name
cluster_data = aid_df[['INSTNM', 'COSTT4_A', 'PCTPELL', 'PCTFLOAN', 'C150_4', 'CDR3']].dropna()
features = cluster_data[['COSTT4_A', 'PCTPELL', 'PCTFLOAN', 'C150_4', 'CDR3']]

# Step 2: Standardize the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Step 3: Apply K-Means clustering
kmeans = KMeans(n_clusters=3, random_state=42)
clusters = kmeans.fit_predict(scaled_features)

# Step 4: Attach clusters to dataframe
cluster_data['CLUSTER'] = clusters

# Step 5: Sort clusters by graduation rate and assign readable labels
grad_means = cluster_data.groupby('CLUSTER')['C150_4'].mean().sort_values()
cluster_name_map = {
    grad_means.index[0]: 'High-Risk, High-Aid Institutions',
    grad_means.index[1]: 'Middle-Tier, Mixed-Aid Institutions',
    grad_means.index[2]: 'Elite, Low-Aid Institutions'
}
ordered_labels = list(cluster_name_map.values())

# Step 6: Assign labels to each row
cluster_data['CLUSTER LABEL'] = cluster_data['CLUSTER'].map(cluster_name_map)
cluster_data['CLUSTER LABEL'] = cluster_data['CLUSTER LABEL'].astype(CategoricalDtype(ordered_labels, ordered=True))

# Step 7: Rename columns for clarity
cluster_data = cluster_data.rename(columns={
    'COSTT4_A': 'Total Cost ($)',
    'PCTPELL': '% Pell Grants',
    'PCTFLOAN': '% Federal Loans',
    'C150_4': 'Graduation Rate',
    'CDR3': 'Default Rate'
})

# Step 8: Save cluster summary (with observed=True to silence warning)
cluster_summary = cluster_data.groupby('CLUSTER LABEL', observed=True)[[
    'Total Cost ($)', '% Pell Grants', '% Federal Loans', 'Graduation Rate', 'Default Rate'
]].mean().round(2)
display(cluster_summary)

# Step 9: Scatter plot
fig = px.scatter(
    cluster_data,
    x='Total Cost ($)',
    y='Graduation Rate',
    color='CLUSTER LABEL',
    title='Institution Clusters: Cost vs Graduation Rate',
    labels={
        'Total Cost ($)': 'Total Cost of Attendance ($)',
        'Graduation Rate': 'Graduation Rate'
    },
    hover_data=['INSTNM', '% Pell Grants', '% Federal Loans', 'Default Rate']
)
fig.update_layout(legend_title_text='Cluster Type')
fig.show()

# Step 10: Final cluster_df output for merging if needed
cluster_df = cluster_data[['INSTNM', 'CLUSTER', 'CLUSTER LABEL']]


,Total Cost ($),% Pell Grants,% Federal Loans,Graduation Rate,Default Rate
CLUSTER LABEL,,,,,
"High-Risk, High-Aid Institutions",35467.04,0.49,0.65,0.43,0.0
"Middle-Tier, Mixed-Aid Institutions",21279.95,0.35,0.26,0.44,0.0
"Elite, Low-Aid Institutions",57398.00,0.23,0.46,0.73,0.0


In [119]:
# Merge if cluster info not already present
if 'CLUSTER' not in aid_df.columns:
    aid_df = aid_df.merge(cluster_df, on='INSTNM', how='left')

if 'CLUSTER LABEL' not in aid_df.columns:
    aid_df['CLUSTER LABEL'] = aid_df['CLUSTER'].map(cluster_name_map).fillna('Unclustered')


In [120]:
# Select and rename for display
display_cols = [
    'INSTNM', 'CITY', 'STABBR', 'ZIP',
    'CLUSTER LABEL',
    'TUITIONFEE_IN', 'TUITIONFEE_OUT',
    'COSTT4_A', 'NET_PRICE',
    'PCTPELL', 'PCTFLOAN',
    'C150_4', 'CDR3'
]

# Filter and rename
cluster_table = aid_df[[col for col in display_cols if col in aid_df.columns]].dropna(subset=['CLUSTER LABEL']).copy()
cluster_table = cluster_table.rename(columns={
    'INSTNM': 'Institution Name',
    'CITY': 'City',
    'STABBR': 'State',
    'ZIP': 'ZIP Code',
    'CLUSTER LABEL': 'Cluster',
    'TUITIONFEE_IN': 'In-State Tuition',
    'TUITIONFEE_OUT': 'Out-of-State Tuition',
    'COSTT4_A': 'Total Cost ($)',
    'NET_PRICE': 'Net Price',
    'PCTPELL': '% Receiving Pell Grants',
    'PCTFLOAN': '% Receiving Federal Loans',
    'C150_4': 'Graduation Rate (4-Year)',
    'CDR3': '3-Year Default Rate'
})

cluster_table.to_csv(data_path + "clustered_institutions.csv", index=False)



In [121]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Define input widgets
zip_input = widgets.Text(value='', placeholder='e.g. 20001', description='ZIP:')
state_input = widgets.Text(value='', placeholder='e.g. DC', description='State:')
city_input = widgets.Text(value='', placeholder='e.g. Washington', description='City:')
school_input = widgets.Text(value='', placeholder='e.g. Howard', description='School:')
cluster_dropdown = widgets.Dropdown(
    options=[''] + sorted(cluster_table['Cluster'].dropna().unique()),
    description='Cluster:'
)

output = widgets.Output()

# Filtering logic
def multi_filter(zip_code, state, city, school, cluster):
    with output:
        clear_output()
        df = cluster_table.copy()

        if zip_code.strip():
            df = df[df['ZIP Code'].astype(str).str.startswith(zip_code.strip()[:3])]
        if state.strip():
            df = df[df['State'].str.contains(state.strip(), case=False, na=False)]
        if city.strip():
            df = df[df['City'].str.contains(city.strip(), case=False, na=False)]
        if school.strip():
            df = df[df['Institution Name'].str.contains(school.strip(), case=False, na=False)]
        if cluster:
            df = df[df['Cluster'] == cluster]

        if df.empty:
            print("❌ No matching institutions found.")
        else:
            display(df.sort_values(by='Institution Name').reset_index(drop=True))

# Display the widget interface
ui = widgets.VBox([school_input, city_input, state_input, zip_input, cluster_dropdown])
interactive_output = widgets.interactive_output(multi_filter, {
    'zip_code': zip_input,
    'state': state_input,
    'city': city_input,
    'school': school_input,
    'cluster': cluster_dropdown
})

display(ui, interactive_output, output)


Output()

Output()

In [122]:
# Check your actual column names first
print(cluster_data.columns)


Index(['INSTNM', 'Total Cost ($)', '% Pell Grants', '% Federal Loans',
       'Graduation Rate', 'Default Rate', 'CLUSTER', 'CLUSTER LABEL'],
      dtype='object')


In [123]:
import plotly.express as px

# Step 1: Map cluster names to integer codes
cluster_order = {
    'High-Risk, High-Aid Institutions': 0,
    'Middle-Tier, Mixed-Aid Institutions': 1,
    'Elite, Low-Aid Institutions': 2
}
cluster_table['Cluster Code'] = cluster_table['Cluster'].map(cluster_order)

# Step 2: Define correct column names from cluster_table
dimensions = [
    'Total Cost ($)',
    '% Receiving Pell Grants',
    '% Receiving Federal Loans',
    'Graduation Rate (4-Year)',
    '3-Year Default Rate'
]

# Step 3: Map display labels
labels = {
    'Cluster Code': 'Cluster',
    'Total Cost ($)': 'Total Cost ($)',
    '% Receiving Pell Grants': '% Pell Grants',
    '% Receiving Federal Loans': '% Federal Loans',
    'Graduation Rate (4-Year)': 'Graduation Rate',
    '3-Year Default Rate': 'Default Rate'
}

# Step 4: Plot
fig = px.parallel_coordinates(
    cluster_table,
    color='Cluster Code',
    dimensions=dimensions,
    color_continuous_scale=[
        [0.0, 'red'],
        [0.5, 'orange'],
        [1.0, 'green']
    ],
    labels=labels
)

fig.update_layout(
    title="📊 Parallel Coordinates: Institutional Clusters",
    margin=dict(t=90, l=80, r=40, b=40),
    coloraxis_colorbar=dict(
        tickvals=[0, 1, 2],
        ticktext=list(cluster_order.keys()),
        title='Cluster'
    )
)

fig.show()



In [140]:
import pandas as pd

# === Step 1: Load Data ===
# Load your institution data (from Colab or your CSV export)
cluster_table = pd.read_csv(data_path + "clustered_institutions.csv")

# Load ZIP-to-lat/lon reference data
zip_data = pd.read_csv(data_path + "uszips.csv")  # Make sure this contains 'zip', 'lat', 'lng' columns

# === Step 2: Clean and Standardize ZIP Codes ===
# Extract first 5-digit sequence from ZIP Code column, in case of ZIP+4 formats like '35294-0110'
cluster_table['ZIP Code'] = (
    cluster_table['ZIP Code']
    .astype(str)
    .str.extract(r'(\d{5})')[0]
)

# Ensure ZIP codes in reference table are also 5-character strings
zip_data['zip'] = zip_data['zip'].astype(str).str.zfill(5)

# === Step 3: Merge Data ===
# Perform left join to retain all institutions and attach lat/lon from ZIP table
merged_df = pd.merge(
    cluster_table,
    zip_data[['zip', 'lat', 'lng']],
    left_on='ZIP Code',
    right_on='zip',
    how='left'
)

# === Step 4: Clean Up Columns ===
# Rename for clarity and drop redundant ZIP field from the reference file
merged_df.rename(columns={'lat': 'Latitude', 'lng': 'Longitude'}, inplace=True)
merged_df.drop(columns=['zip'], inplace=True)

# === Step 5: Export Final Clean File ===
# Save to a new CSV to load into Tableau
merged_df.to_csv(data_path + "clustered_institutions_with_coords.csv", index=False)

# Optional: check how many ZIPs were successfully matched
num_matched = merged_df['Latitude'].notna().sum()
num_total = len(merged_df)
print(f"✅ Latitude/Longitude matched for {num_matched} of {num_total} institutions.")



✅ Latitude/Longitude matched for 1943 of 2040 institutions.


## Predictive Modeling: Graduation Rate Estimation

Using a linear regression model, we attempt to predict graduation rates based on financial inputs such as tuition, Pell Grant share, and loan reliance. Model accuracy and feature importance are examined.

**Model Outputs:**
- R² score and mean squared error
- Standardized coefficients for feature interpretation
- Plots: actual vs. predicted, residual distribution


In [124]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Step 1: Select features and target from cluster_table
regression_df = cluster_table.dropna(subset=[
    'Total Cost ($)', '% Receiving Pell Grants', '% Receiving Federal Loans', 'Graduation Rate (4-Year)'
])

X = regression_df[['Total Cost ($)', '% Receiving Pell Grants', '% Receiving Federal Loans']]
y = regression_df['Graduation Rate (4-Year)']

# Step 2: Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 3: Train model
model = LinearRegression()
model.fit(X_train, y_train)

# Step 4: Predict and evaluate
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f'📊 R² Score: {r2:.2f}')
print(f'📉 Mean Squared Error: {mse:.4f}')
print(f'🔢 Intercept: {model.intercept_:.4f}')

# Step 5: Coefficient table
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).round(4)

coefficients




📊 R² Score: 0.42
📉 Mean Squared Error: 0.0228
🔢 Intercept: 0.4519


,Feature,Coefficient
0,Total Cost ($),0.0000
1,% Receiving Pell Grants,-0.2648
2,% Receiving Federal Loans,-0.0729


In [125]:
import plotly.express as px
import pandas as pd

# Create a DataFrame for plotting
regression_results = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred
})

fig = px.scatter(
    regression_results,
    x='Actual',
    y='Predicted',
    title='Actual vs. Predicted Graduation Rate',
    labels={'Actual': 'Actual Graduation Rate', 'Predicted': 'Predicted Graduation Rate'},
    trendline="ols",
    opacity=0.7
)
fig.update_layout(
    width=700,
    height=500,
    showlegend=False
)
fig.show()


In [126]:
residuals = y_test - y_pred
residual_df = pd.DataFrame({'Residuals': residuals})

fig = px.histogram(
    residual_df,
    x='Residuals',
    nbins=30,
    title='Distribution of Residuals',
    labels={'Residuals': 'Prediction Error (Residual)'},
    opacity=0.75
)

fig.update_layout(
    width=700,
    height=500,
    yaxis_title='Frequency'
)

fig.show()



In [127]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import pandas as pd

# Step 1: Select and clean from cluster_table
regression_df = cluster_table.dropna(subset=[
    'Total Cost ($)', '% Receiving Pell Grants', '% Receiving Federal Loans', 'Graduation Rate (4-Year)'
])

X = regression_df[['Total Cost ($)', '% Receiving Pell Grants', '% Receiving Federal Loans']]
y = regression_df['Graduation Rate (4-Year)']

# Step 2: Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 3: Train-test split
X_train_scaled, X_test_scaled, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Step 4: Fit model
reg_std = LinearRegression()
reg_std.fit(X_train_scaled, y_train)

# Step 5: Predict and extract coefficients
y_pred_std = reg_std.predict(X_test_scaled)

std_coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Standardized Coefficient': reg_std.coef_
}).sort_values(by='Standardized Coefficient', key=abs, ascending=False).round(4)

std_coefficients



,Feature,Standardized Coefficient
0,Total Cost ($),0.1047
1,% Receiving Pell Grants,-0.0454
2,% Receiving Federal Loans,-0.0159


In [128]:
# Ensure all rows are complete
regression_df = cluster_table.dropna(subset=[
    'Total Cost ($)', '% Receiving Pell Grants', '% Receiving Federal Loans', 'Graduation Rate (4-Year)'
])

# Split features and target
features = regression_df[['Total Cost ($)', '% Receiving Pell Grants', '% Receiving Federal Loans']]
target = regression_df['Graduation Rate (4-Year)']

# Optional: Use the same regressor if not already defined
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

regressor = LinearRegression()
cv_r2_scores = cross_val_score(regressor, features, target, cv=5, scoring='r2')
print(f'Cross-validated R² scores: {cv_r2_scores}')
print(f'Mean CV R²: {cv_r2_scores.mean():.2f}')


Cross-validated R² scores: [0.46117969 0.43484083 0.43957036 0.53395171 0.04663729]
Mean CV R²: 0.38


In [129]:
import plotly.express as px

# Define friendly label mapping
label_map = {
    'COSTT4_A': 'Total Cost ($)',
    'PCTPELL': '% Receiving Pell Grants',
    'PCTFLOAN': '% Receiving Federal Loans'
}

# Apply mapping to the y-axis labels
std_coefficients['Feature Label'] = std_coefficients['Feature'].map(label_map)

# Plot with updated labels
fig = px.bar(
    std_coefficients,
    x='Standardized Coefficient',
    y='Feature Label',
    orientation='h',
    title='Feature Importance: Predictors of Graduation Rate',
    color='Standardized Coefficient',
    color_continuous_scale='Tealrose',
    labels={'Feature Label': 'Feature'}
)

fig.update_layout(
    width=700,
    height=400,
    yaxis=dict(categoryorder='total ascending')
)

fig.show()



### Regression Analysis Insights: Predicting Graduation Rates

This section used a multiple linear regression model to predict college graduation rates (`C150_4`) based on:

- Total Cost of Attendance (`COSTT4_A`)
- Percent of Students Receiving Pell Grants (`PCTPELL`)
- Percent of Students Receiving Federal Loans (`PCTFLOAN`)

---

#### Model Performance
- **R² Score**: The model explained approximately **{r2:.2f}** of the variance in graduation rates.
- **Mean Squared Error**: Provided an average squared deviation between actual and predicted graduation rates.

---

#### Key Findings
- **Total Cost (`COSTT4_A`)** had a **positive effect** on graduation rate. While high cost doesn't cause better outcomes, it may signal more resources or selectivity.
- **Percent Pell (`PCTPELL`)** and **Percent Federal Loans (`PCTFLOAN`)** both had **negative coefficients**, suggesting institutions with more economically disadvantaged students may have lower graduation rates.
- These relationships may reflect systemic barriers and institutional disparities in support services, not individual capability.

---

#### Interpretation
This regression model highlights how institutional characteristics tied to **student financial aid** correlate with student success outcomes. While the model doesn’t imply causation, it reveals patterns useful for policy, intervention targeting, or further study.

For a more robust analysis, consider incorporating:
- Regional or demographic breakdowns
- Institutional control (public/private/for-profit)
- Time-series trends

---


## Additional Insights: Loan Dependency and Risk

In this section, we examine how loan dependency relates to risk indicators like cohort default rate. These insights can help identify institutions or populations needing additional support.

**Key Question:**
- Does a higher reliance on federal loans correlate with worse repayment outcomes?


In [130]:
import plotly.express as px

# Ensure INSTITUTION_TYPE is available
if 'INSTITUTION_TYPE' not in aid_df.columns and 'CONTROL' in aid_df.columns:
    control_map = {1: 'Public', 2: 'Private Non-Profit', 3: 'Private For-Profit'}
    aid_df['INSTITUTION_TYPE'] = aid_df['CONTROL'].map(control_map)

# Prepare clean data
loan_default_df = aid_df[['INSTITUTION_TYPE', 'PCTFLOAN', 'CDR3']].copy().dropna()

# Correlation
correlation = loan_default_df['PCTFLOAN'].corr(loan_default_df['CDR3'])
print(f"📉 Correlation between % Loan Dependency and Default Rate: {correlation:.2f}")

# Plot
fig = px.scatter(
    loan_default_df,
    x='PCTFLOAN',
    y='CDR3',
    color='INSTITUTION_TYPE',
    title='Loan Dependency vs. Default Rate by Institution Type',
    labels={
        'PCTFLOAN': '% Students with Federal Loans',
        'CDR3': '3-Year Cohort Default Rate',
        'INSTITUTION_TYPE': 'Institution Type'
    },
    opacity=0.6
)
fig.update_layout(yaxis_range=[0, 0.2])
fig.show()


📉 Correlation between % Loan Dependency and Default Rate: -0.06


## Conclusion & Recommendations

This analysis highlights several important takeaways:

- Public institutions generally offer lower tuition and net price.
- Pell Grant and loan dependency vary widely by institution type.
- Graduation outcomes correlate modestly with cost and strongly with student aid profiles.
- Predictive models can identify which financial factors most influence student success.

**Next Steps:**
- Explore regional or demographic disaggregation
- Investigate institutional policies that buffer risk
- Apply findings to inform financial aid reform or targeted outreach
